# 🎯 YOLOX-S — Détection de cibles radar (Kaggle)

Ce notebook entraîne le modèle YOLOX-S sur le dataset SOC_50classes depuis Kaggle (GPU T4/P100).

**Pré-requis avant d'exécuter :**
1. **GPU activé** : Settings → Accelerator → GPU T4 x2 (ou P100)
2. **Internet activé** : Settings → Internet → On
3. **Token GitHub** ajouté en secret Kaggle : Add-ons → Secrets → `GITHUB_TOKEN`
4. **Dataset attaché** : Ajouter le dataset `SOC_50classes_coco` via le menu + (voir cellule de configuration)

**Étapes :**
1. Vérification du GPU
2. Configuration
3. Vérification du dataset Kaggle
4. Clone du projet et installation des dépendances
5. Lancement de l'entraînement
6. Sauvegarde des checkpoints

## 1. Vérification du GPU

In [ ]:
import torch

print(f'GPU disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go')
    print(f'PyTorch : {torch.__version__}')
else:
    print('⚠️  Pas de GPU détecté. Aller dans Settings → Accelerator → GPU T4 x2')

## 2. Configuration

Adapter `DATASET_SLUG` au slug de votre dataset Kaggle (le nom qui apparaît dans l'URL de votre dataset).

In [ ]:
# ── À ADAPTER ──────────────────────────────────────────────────────────────
DATASET_SLUG  = 'SOC_50classes_coco'    # slug du dataset Kaggle (sans le préfixe username/)
BRANCH        = 'mael'                  # branche GitHub
# ───────────────────────────────────────────────────────────────────────────

# Paramètres d'entraînement
MAX_EPOCH  = 100
BATCH_SIZE = 32
RUN_NAME   = 'yolox_s_kaggle_v1'
FP16       = True

# Chemins fixes Kaggle
PROJECT_DIR  = '/kaggle/working/radar_detection'
DATASET_PATH = f'/kaggle/input/{DATASET_SLUG}'

print(f'Dataset      : {DATASET_PATH}')
print(f'Projet       : {PROJECT_DIR}')
print(f'Max epochs   : {MAX_EPOCH}')
print(f'Batch size   : {BATCH_SIZE}')
print(f'FP16         : {FP16}')

## 3. Vérification du dataset Kaggle

Le dataset doit être attaché à ce notebook et avoir la structure suivante :
```
/kaggle/input/{DATASET_SLUG}/
└── SOC_50classes_coco/
    ├── annotations/
    │   ├── train.json
    │   └── test.json
    └── images/
        ├── train/
        └── test/
```
Pour attacher le dataset : cliquez sur **+** (Add data) en haut à droite.

In [ ]:
import os
from pathlib import Path

assert os.path.exists(DATASET_PATH), (
    f"Dataset introuvable : {DATASET_PATH}\n"
    f"→ Attachez le dataset '{DATASET_SLUG}' via le menu + (Add data)"
)

# Afficher la structure
!find {DATASET_PATH} -maxdepth 3 -type d

# Compter les images
train_imgs = list(Path(f'{DATASET_PATH}/SOC_50classes_coco/images/train').iterdir())
test_imgs  = list(Path(f'{DATASET_PATH}/SOC_50classes_coco/images/test').iterdir())
print(f'\nTrain images : {len(train_imgs)}')
print(f'Test  images : {len(test_imgs)}')

## 4. Clone du projet et installation des dépendances

Le token GitHub doit être ajouté dans **Add-ons → Secrets** sous le nom `GITHUB_TOKEN`.

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets  = UserSecretsClient()
GITHUB_TOKEN  = user_secrets.get_secret('GITHUB_TOKEN')
REPO_URL      = f'https://{GITHUB_TOKEN}@github.com/william94000schr/ATR_SAR.git'

!rm -rf {PROJECT_DIR}
!git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}
os.chdir(PROJECT_DIR)
print(f'Répertoire de travail : {os.getcwd()}')
!ls

In [ ]:
# Créer le lien symbolique dataset → chemin attendu par la config
!mkdir -p data

data_link = Path('data/SOC_50classes_coco')
if data_link.is_symlink():
    data_link.unlink()

!ln -sf {DATASET_PATH} data/SOC_50classes_coco

!echo "Train images :" $(ls data/SOC_50classes_coco/SOC_50classes_coco/images/train | wc -l)
!echo "Test  images :" $(ls data/SOC_50classes_coco/SOC_50classes_coco/images/test  | wc -l)
print('✅ Dataset lié')

In [ ]:
# Installer YOLOX sans ses dépendances problématiques
!pip install -q --no-deps yolox

# Dépendances nécessaires
!pip install -q loguru thop ninja tabulate

import yolox
print(f'✅ YOLOX installé : {yolox.__version__}')

In [ ]:
# Vérification rapide d'une image
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

sample = next(Path('data/SOC_50classes_coco/SOC_50classes_coco/images/train').glob('*.tif'))
img = Image.open(sample)
arr = np.array(img)
print(f'Fichier : {sample.name}')
print(f'Taille : {img.size}, Mode : {img.mode}, dtype : {arr.dtype}')
print(f'Min/Max : {arr.min()} / {arr.max()}')

plt.figure(figsize=(4, 4))
plt.imshow(arr, cmap='gray')
plt.title(sample.name)
plt.axis('off')
plt.show()

## 5. Lancement de l'entraînement

In [ ]:
fp16_flag = '--fp16' if FP16 else '--no-fp16'
print(f'Config : {MAX_EPOCH} epochs, batch={BATCH_SIZE}, fp16={FP16}')

!python train.py \
    --config config/yolo_s.py \
    --max-epoch {MAX_EPOCH} \
    --batch-size {BATCH_SIZE} \
    --run-name {RUN_NAME} \
    {fp16_flag}

## 6. Sauvegarde des checkpoints

Les fichiers dans `/kaggle/working/` sont automatiquement sauvegardés comme outputs du kernel.
Vous pouvez aussi télécharger l'archive `results.zip` générée ci-dessous.

In [ ]:
import glob
import shutil

# Lister les checkpoints
checkpoints = sorted(glob.glob('outputs/**/*.pth', recursive=True))
if checkpoints:
    print('Checkpoints :')
    for c in checkpoints:
        size_mb = os.path.getsize(c) / 1e6
        print(f'  {c}  ({size_mb:.1f} Mo)')
else:
    print('⚠️  Aucun checkpoint trouvé.')

# Copier les outputs dans /kaggle/working/ pour sauvegarde automatique
dest = Path('/kaggle/working/outputs')
if dest.exists():
    shutil.rmtree(dest)
shutil.copytree('outputs', dest)
print(f'\n✅ Checkpoints copiés dans {dest}')

# Créer une archive zip pour téléchargement direct
shutil.make_archive('/kaggle/working/results', 'zip', 'outputs')
print('✅ Archive : /kaggle/working/results.zip')